# 1. nn.Module
It's the basis for neural networks in PyTorch. nn.Module it's the mother class, the we gonna use the POO to inherit the functions that the mother classe have

1.1 This are some **dunder methods**:
- ``__init()``
- `forward()`

1.2 We also have the intern dicts, that are for we organize everthing that we put int he __init__, like:
- `._parameters`
- `._modules`
- `._buffers`
- `named_parameters()` and more
- `state_dict()`

1.3 The weights and model, we can save the model and the weights to fine tunning and more
- `state_dict()`

### 1.1 Dunder Methods

In [5]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.l1 = nn.Sequential(
            nn.Linear(5, 10, bias = False),
            nn.ReLU()
            )
        
        self.l2 = nn.Sequential(
            nn.Linear(10, 5, bias = False),
            nn.ReLU()
            )
        
    def forward(self, x):
        print(f'Passing by the forward without calling!')
        return self.l2(self.l1(x))
    
model = Model()

try1 = model(torch.rand(10, 5))
print(try1)


Passing by the forward without calling!
tensor([[0.1218, 0.1200, 0.2944, 0.1873, 0.0916],
        [0.1510, 0.2031, 0.3725, 0.2706, 0.1411],
        [0.2477, 0.2502, 0.4606, 0.2858, 0.1555],
        [0.1292, 0.1244, 0.2881, 0.1846, 0.0816],
        [0.2017, 0.2160, 0.4558, 0.3092, 0.1455],
        [0.2158, 0.2183, 0.4579, 0.3060, 0.1484],
        [0.2670, 0.2897, 0.5475, 0.3596, 0.1569],
        [0.1406, 0.1896, 0.2756, 0.2107, 0.1523],
        [0.2283, 0.2127, 0.5051, 0.3351, 0.1565],
        [0.1712, 0.1701, 0.3179, 0.2201, 0.1207]], grad_fn=<ReluBackward0>)


### 1.2 Intern Dicts

In [1]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        pass

model = Model()
print(f'Parameters: {model._parameters}')
print(f'Modules: {model._modules}')
print(f'Buffers: {model._buffers}')

Parameters: {}
Modules: {}
Buffers: {}


#### 1.2.1 Parameteres
A function that add and transform data into parameters `nn.Parameters()`

In [2]:
import torch

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.parameter1 = nn.Parameter(torch.tensor(4.0))


model = Model()
print(f'Parameters: {model._parameters}')

Parameters: {'parameter1': Parameter containing:
tensor(4., requires_grad=True)}


### 1.2.2 Modules
Are the layers of the model (nn.Linear, nn.Conv2d and more)



In [11]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer1 = nn.Linear(1,1 )


model = Model()
print(f'Modules: {model._modules}')

Modules: {'layer1': Linear(in_features=1, out_features=1, bias=True)}


#### 1.2.3 Buffers
Are the the place where functions of the model that don't have treinable parameters are allocated (Normalization, Masks and more) .register_buffer()


In [10]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.constant = torch.tensor(1.0)
        self.register_buffer('my constant' , self.constant)

model = Model()
print(f'Buffers: {model._buffers}')

Buffers: {'my constant': tensor(1.)}


> WARNING!!
PyTorch Internal Structure: nn.Module, Parameters and Buffers
To understand how PyTorch manages models, it`s essential to distinguish between the objects you define (modules) and the data they contain (parameters and buffers).

The Hierarchy of nn.Module In PyTorch, everything that makes up your model is organized into internal "drawers" inside an nn.Module object. When you create a class, PyTorch manages these drawers automatically:

``._modules``: Contains sub-modules defined as attributes.

``._parameters``: Contains tensors that you manually set to nn.Parameter.

``._buffers``: Contains tensors that you manually registered as buffers.

Where should each thing stay?

### A. Occupying the ._modules (The most common path) The vast majority of components of a neural network (such as nn.Linear, nn.Conv2d, nn.BatchNorm1d) must be assigned directly as an attribute of its class in __init__.

Why? By doing `self.fc = nn.Linear(...)`, PyTorch detects this object and automatically places it in ._modules.
Result: PyTorch "opens" this module, finds the weights and biases within it, and manages everything automatically (saving, loading, moving between CPU/GPU, and calculating gradients).

### B. Using ._parameters (Manual Definition) You should only add items to ._parameters if you are creating a custom layer from scratch and need a tensor that should be updated by the optimizer (via gradient), but is not a ready-made module.
- How to: ``self.my_param = nn.Parameter(torch.randn(10, 10))``
- Result: PyTorch identifies that this is a trainable parameter and includes it in the optimizer calculation.

### C. Using ._buffers (Non-Trainable State) You should use buffers for data that is part of the model`s "state" but should not be changed by the optimizer.
- As: ``self.register_buffer(name, tensor)``}
Result: PyTorch knows that this tensor needs to be saved in state_dict and moved along with the model (e.g. to(cuda)), but the optimizer will not try to apply gradients over it.
Why dont you see the parameters inside self._parameters? If you set self.fc = nn.Linear(10, 5), the weights (weight) and bias (bias) of the fclayer **will not** be in theself._parametersof your main class. They will be inside the_parameters drawer of the **self.fc` object**.
To visualize everything in an organized way, PyTorch offers recursive methods:

For see in the normal form:

1. ``model.named_parameters()``: Lists all parameters of the root and all submodules
2. ``model.named_modules()``: List all the modules in the model
3. ``model.named_buffers()``: Lists all buffers for the root and all submodules. 

In [6]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ParameterOwn = nn.Parameter(torch.tensor(2.0))
        self.l1 = nn.Linear(3,3)
        self.bn = nn.BatchNorm1d(num_features=2)

model = Model()

print(f'Parameters:')
for name, param in model.named_parameters():
    print(f"Name: {name} | Size: {param.shape} | Trainable?: {param.requires_grad}")

print(f'\nModules:')
for name, module in model.named_modules():
    print(f"`Path: {name} | Module: {module}")

print(f'\nBuffers:')
for name, buffer in model.named_buffers():
    print(f"Name: {name} | Size: {buffer.shape}")

Parameters:
Name: ParameterOwn | Size: torch.Size([]) | Trainable?: True
Name: l1.weight | Size: torch.Size([3, 3]) | Trainable?: True
Name: l1.bias | Size: torch.Size([3]) | Trainable?: True
Name: bn.weight | Size: torch.Size([2]) | Trainable?: True
Name: bn.bias | Size: torch.Size([2]) | Trainable?: True

Modules:
`Path:  | Module: Model(
  (l1): Linear(in_features=3, out_features=3, bias=True)
  (bn): BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)
`Path: l1 | Module: Linear(in_features=3, out_features=3, bias=True)
`Path: bn | Module: BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)

Buffers:
Name: bn.running_mean | Size: torch.Size([2])
Name: bn.running_var | Size: torch.Size([2])
Name: bn.num_batches_tracked | Size: torch.Size([])


## 1.3 Acess the Weights and Model
Now that we see how to saw and understand the parameters, modules and buffers from the model. We need to have the acess of that values. Using the:


Get:

`state_dict()`: Collects all parameters (weights/bias) and all buffers (moving averages, etc.) from the entire model, including submodules, into a single Python dictionary.

`torch.save(thing, .name)`: will create a file . something (normally .safetensors but for here i'll use the .pth)

> You can use `torch.save(model, 'model.pth')` or `torch.save(model.state_dict(), 'weights.pth')`. With the weights and can do the fine tunning, with the model we have the model for inference

Use:

`torch.load()`: Do the PyTorch read things that isn't in the natural language of the framework

`model.load_state_dict()`: 
> When use the `.load_state_dict()` the architecture between both models need to be the same, or an error gonna occurs. To do some fine tunning it's better to use manual techniques 


In [17]:
state = model.state_dict()
l1 = state['l1.weight']

print(f'All the weights: {state}')
print(f'Acess the L1 Weights: {l1}')

All the weights: OrderedDict({'ParameterOwn': tensor(2.), 'l1.weight': tensor([[-0.0743, -0.2999, -0.5577],
        [ 0.4297,  0.1237, -0.1074],
        [ 0.2447,  0.4287, -0.4240]]), 'l1.bias': tensor([-0.0247, -0.4289,  0.0075]), 'bn.weight': tensor([1., 1.]), 'bn.bias': tensor([0., 0.]), 'bn.running_mean': tensor([0., 0.]), 'bn.running_var': tensor([1., 1.]), 'bn.num_batches_tracked': tensor(0)})
Acess the L1 Weights: tensor([[-0.0743, -0.2999, -0.5577],
        [ 0.4297,  0.1237, -0.1074],
        [ 0.2447,  0.4287, -0.4240]])


In [23]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.l1 = nn.Sequential(
            nn.Linear(5, 10, bias = False),
            nn.ReLU(),
            nn.Linear(10, 1, bias = False)
        )

model1 = Model()

# Saving the weights
state_dict = model1.state_dict()
torch.save(state_dict, 'weights_model1.pth')
print(f'Weights of the first model: \n{state_dict}\n\n\n')


# Model 2
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.l1 = nn.Sequential(
            nn.Linear(5, 10, bias = False),
            nn.ReLU(),
            nn.Linear(10, 1, bias = False)
        )
model2 = Model()

# Saving the weights
state_dict = model2.state_dict()
print(f'Weights of the second model: \n{state_dict}\n\n\n')

# Change the weights from model2
weights_model1 = torch.load('weights_model1.pth')
model2.load_state_dict(weights_model1)

# Get the news weights
state_dict = model2.state_dict()


# Visualizing
print(f'Weights of the second model (update version): \n{state_dict}')




Weights of the first model: 
OrderedDict({'l1.0.weight': tensor([[-0.1553, -0.3754,  0.0858, -0.3887,  0.2157],
        [ 0.3388,  0.1418,  0.0639, -0.1578, -0.1920],
        [ 0.1643, -0.2110, -0.1792, -0.3715,  0.4194],
        [ 0.2649, -0.0141,  0.1176, -0.3961,  0.2391],
        [-0.2904, -0.3711, -0.0646, -0.3704, -0.0563],
        [-0.4465,  0.2951, -0.0155,  0.0502,  0.2635],
        [-0.2100,  0.2667, -0.3194, -0.0621,  0.2595],
        [ 0.2071,  0.0079, -0.2473, -0.2111,  0.4254],
        [-0.1031, -0.4395, -0.1898, -0.3065,  0.3027],
        [ 0.2599,  0.1158, -0.4289,  0.3017,  0.2695]]), 'l1.2.weight': tensor([[-0.3001, -0.2412, -0.2632,  0.0891,  0.2542, -0.1927, -0.1597,  0.0056,
          0.0876, -0.2123]])})



Weights of the second model: 
OrderedDict({'l1.0.weight': tensor([[ 0.0308, -0.0965, -0.4380, -0.2830,  0.2625],
        [ 0.1128,  0.0240,  0.2223, -0.1583, -0.0036],
        [ 0.0324,  0.2716,  0.0480, -0.3543, -0.4299],
        [ 0.1587, -0.3147,  0.0277, -0

#### Model Weight Formats: .pth vs .safetensors
When working with PyTorch, you will primarily encounter two file formats for storing your model weights. Here is the breakdown of the differences.

1. `.pth` (PyTorch Legacy)
- Technology: Uses Python's pickle module by default.
- Pros: Very easy to use with torch.save() and torch.load(); supports saving complex Python objects (not just tensors).
- Cons: * Security Risk: Because it uses pickle, it can execute arbitrary code during loading. Never load a .pth file from an untrusted source.
    - Performance: Slower to load because it requires unpickling and conversion.
    - Coupling: Highly tied to the PyTorch ecosystem.

2. `.safetensors` (The Modern Standard)
- Technology: A format developed by Hugging Face designed specifically for storing tensors.
- Pros:
    - Security: "Zero-code" format. It stores only data (tensors), making it impossible to execute malicious code upon loading.
    - Performance: Optimized for speed using memory-mapping (mmap). It allows for near-instant loading of large models.
    - Framework Agnostic: Can be read by various libraries (PyTorch, TensorFlow, JAX) without needing to re-convert.
Cons: Only stores tensors; it cannot store complex Python objects (like custom classes or arbitrary dictionaries).